# 18.9 综合实践一：编辑候选的多层证据与优先级

《统计遗传与智能育种学》第十八章 · 电子资源 `18_design_to_edit.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：18_design_to_edit.ipynb。实践生成120个候选变异，分别记录统计关联、共定位、自然等位基因方向一致性、序列模型效应、保守性、可编辑性、多效性风险和预测脱靶风险。代码先执行预先声明的最低门槛，再用固定基础权重计算验证优先级；随后进行2000次权重扰动，计算每个候选进入前10名（Top10）的频率。

运行检查：重启内核并运行全部单元。本书实际运行中，120个候选有71个通过最低门槛，但只有5个候选在2000次权重扰动中以至少80%的频率进入前10名（Top10）。必须同时报告基础得分与排序稳定性，并明确两者都不是因果概率。

## 输入、完整代码、实际输出与解释

输入与可复现条件：SEED=1801；120个模拟候选；8类证据/风险特征；固定门槛与基础权重；2000次Dirichlet分布权重扰动；无外部数据文件。

In [1]:
import sys
import numpy as np
import pandas as pd

SEED = 1801
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('numpy', np.__version__, 'pandas', pd.__version__)
print('SEED', SEED)

n = 120
cand = pd.DataFrame({
    'variant': [f'v{i:03d}' for i in range(n)],
    'association': rng.beta(2.2, 2.0, n),
    'colocalization': rng.beta(1.8, 2.3, n),
    'allelic_coherence': rng.beta(2.0, 2.0, n),
    'sequence_effect': rng.beta(1.7, 2.2, n),
    'conservation': rng.beta(2.0, 2.0, n),
    'editability': rng.beta(2.4, 1.8, n),
    'pleiotropy_risk': rng.beta(1.5, 3.2, n),
    'offtarget_risk': rng.beta(1.4, 3.5, n),
})
benefit = ['association', 'colocalization', 'allelic_coherence',
           'sequence_effect', 'conservation', 'editability']
risk = ['pleiotropy_risk', 'offtarget_risk']
features = benefit + risk
X = cand[features].copy()
for c in risk:
    X[c] = 1 - X[c]
base_w = np.array([.20, .16, .14, .12, .08, .12, .10, .08])
cand['priority_score'] = X.to_numpy() @ base_w
cand['passes_gate'] = ((cand.association >= .35)
                       & (cand.editability >= .35)
                       & (cand.offtarget_risk <= .60))
eligible = np.flatnonzero(cand.passes_gate.to_numpy())
top_count = np.zeros(n, dtype=int)
for _ in range(2000):
    w = rng.dirichlet(base_w * 60)
    score = X.to_numpy() @ w
    k = min(10, len(eligible))
    chosen = eligible[np.argsort(score[eligible])[-k:]]
    top_count[chosen] += 1
cand['top10_stability'] = top_count / 2000
show = cand[cand.passes_gate].sort_values(
    ['top10_stability', 'priority_score'], ascending=False).head(10)
print('eligible_candidates', int(cand.passes_gate.sum()), 'of', n)
print(show[['variant', 'priority_score', 'top10_stability',
            'association', 'colocalization', 'allelic_coherence',
            'sequence_effect', 'editability', 'offtarget_risk']]
      .round(3).to_string(index=False))
print('stable_top10_ge_0.80', int((cand.top10_stability >= .80).sum()))


Python 3.12.13
numpy 2.3.5 pandas 2.2.3
SEED 1801
eligible_candidates 71 of 120
variant  priority_score  top10_stability  association  colocalization  allelic_coherence  sequence_effect  editability  offtarget_risk
   v104           0.733            1.000        0.893           0.471              0.879            0.516        0.890           0.124
   v054           0.740            1.000        0.805           0.537              0.557            0.971        0.923           0.187
   v087           0.705            0.966        0.351           0.616              0.970            0.917        0.931           0.338
   v011           0.697            0.897        0.933           0.642              0.046            0.807        0.725           0.173
   v092           0.673            0.836        0.448           0.863              0.716            0.740        0.439           0.127
   v063           0.661            0.684        0.414           0.841              0.911            0.199     

结果解释：基础综合分最高的候选并不必然具有最高的Top10稳定性。权重敏感性分析用于评价候选排序对主观权重设定的依赖程度，并不能消除不确定性。筛选门槛用于排除明显不适合直接编辑验证的候选，也不能替代统计遗传与功能证据。

可修改练习：把sequence_effect基础权重提高一倍后重新归一化全部权重；再把offtarget_risk门槛从0.60收紧到0.30，比较Top10重叠率、候选数量和稳定性。